# Solving a complex task with a multi-agent hierarchy

This notebook is part of the [Hugging Face Agents Course](https://www.hf.co/learn/agents-course), a free Course from beginner to expert, where you learn to build Agents.

![Agents course share](https://huggingface.co/datasets/agents-course/course-images/resolve/main/en/communication/share.png)

The reception is approaching! With your help, Alfred is now nearly finished with the preparations.

But now there's a problem: the Batmobile has disappeared. Alfred needs to find a replacement, and find it quickly.

Fortunately, a few biopics have been done on Bruce Wayne's life, so maybe Alfred could get a car left behind on one of the movie set, and re-engineer it up to modern standards, which certainly would include a full self-driving option.

But this could be anywhere in the filming locations around the world - which could be numerous.

So Alfred wants your help. Could you build an agent able to solve this task?

> 👉 Find all Batman filming locations in the world, calculate the time to transfer via a cargo plane to there, and represent them on a map, with a color varying by a cargo plane transfer time. Also represent some supercar factories with the same cargo plane transfer time.

Let's build this!

In [2]:
!pip install smolagents[litellm] plotly geopandas shapely kaleido -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires pydantic<=2.12.3,>=2.0, but you have pydantic 2.12.5 which is incompatible.
typer 0.25.1 requires click>=8.2.1, but you have click 8.1.8 which is incompatible.


In [3]:
from huggingface_hub import notebook_login

notebook_login()

In [2]:
# We first make a tool to get the cargo plane transfer time.
import math
from typing import Optional, Tuple

from smolagents import tool


@tool
def calculate_cargo_travel_time(
    origin_coords: Tuple[float, float],
    destination_coords: Tuple[float, float],
    cruising_speed_kmh: Optional[float] = 750.0,  # Average speed for cargo planes
) -> float:
    """
    Calculate the travel time for a cargo plane between two points on Earth using great-circle distance.

    Args:
        origin_coords: Tuple of (latitude, longitude) for the starting point
        destination_coords: Tuple of (latitude, longitude) for the destination
        cruising_speed_kmh: Optional cruising speed in km/h (defaults to 750 km/h for typical cargo planes)

    Returns:
        float: The estimated travel time in hours

    Example:
        >>> # Chicago (41.8781° N, 87.6298° W) to Sydney (33.8688° S, 151.2093° E)
        >>> result = calculate_cargo_travel_time((41.8781, -87.6298), (-33.8688, 151.2093))
    """

    def to_radians(degrees: float) -> float:
        return degrees * (math.pi / 180)

    # Extract coordinates
    lat1, lon1 = map(to_radians, origin_coords)
    lat2, lon2 = map(to_radians, destination_coords)

    # Earth's radius in kilometers
    EARTH_RADIUS_KM = 6371.0

    # Calculate great-circle distance using the haversine formula
    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = (
        math.sin(dlat / 2) ** 2
        + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    )
    c = 2 * math.asin(math.sqrt(a))
    distance = EARTH_RADIUS_KM * c

    # Add 10% to account for non-direct routes and air traffic controls
    actual_distance = distance * 1.1

    # Calculate flight time
    # Add 1 hour for takeoff and landing procedures
    flight_time = (actual_distance / cruising_speed_kmh) + 1.0

    # Format the results
    return round(flight_time, 2)


print(calculate_cargo_travel_time((41.8781, -87.6298), (-33.8688, 151.2093)))

22.82


For the model provider, we use Together AI, one of the new [inference providers on the Hub](https://huggingface.co/blog/inference-providers)!

Regarding the GoogleSearchTool: this requires either having setup env variable `SERPAPI_API_KEY` and passing `provider="serpapi"` or having `SERPER_API_KEY` and passing `provider=serper`.

If you don't have any Serp API provider setup, you can use `WebSearchTool` but beware that it has a rate limit.

In [ ]:
import os
from PIL import Image
from smolagents import CodeAgent, GoogleSearchTool, InferenceClientModel, VisitWebpageTool, LiteLLMModel, DuckDuckGoSearchTool

model = LiteLLMModel(
    model_id="gemini/gemini-3.1-flash-lite",
    api_key="my_gemini_key_was_here"
)

# model = InferenceClientModel(model_id="Qwen/Qwen2.5-Coder-32B-Instruct", provider="together")

We can start with creating a baseline, simple agent to give us a simple report.

In [6]:
task = """Find all Batman filming locations in the world, calculate the time to transfer via cargo plane to here (we're in Gotham, 40.7128° N, 74.0060° W), and return them to me as a pandas dataframe.
Also give me some supercar factories with the same cargo plane transfer time."""

In [10]:
!pip install ddgs

In [ ]:
# from google.colab import userdata
# import os
# os.environ["SERPAPI_API_KEY"] = userdata.get('SERPAPI_API_KEY')

In [11]:
agent = CodeAgent(
    model=model,
    tools=[DuckDuckGoSearchTool(), VisitWebpageTool(), calculate_cargo_travel_time],
    additional_authorized_imports=["pandas"],
    max_steps=20,
)

In [12]:
result = agent.run(task)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Find all Batman filming locations in the world, calculate the time to transfer via cargo plane to here (we're   │
│ in Gotham, 40.7128° N, 74.0060° W), and return them to me as a pandas dataframe.                                │
│ Also give me some supercar factories with the same cargo plane transfer time.                                   │
│                                                                                                                 │
╰─ LiteLLMModel - gemini/gemini-3.1-flash-lite ───────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Error in code parsing:
expected string or bytes-like object, got 'NoneType'
Make sure to provide correct code blobs.

[Step 1: Duration 21.52 seconds| Input tokens: 2,468 | Output tokens: 0]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

...

[Step 4: Duration 7.52 seconds| Input tokens: 12,011 | Output tokens: 865]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 5 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer(df)                                                                                                 
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer:            Category                      Name  Travel Time (h)
0  Filming Location                   Chicago             2.68
1  Filming Location                    London             9.17
2  Filming Location                 Liverpool             8.81
3  Filming Location             Hertfordshire             9.14
4  Supercar Factory       Ferrari (Maranello)            10.72
5  Supercar Factory  Lamborghini (Sant'Agata)            10.74
6  Supercar Factory      Pagani (San Cesario)            10.74

[Step 5: Duration 15.81 seconds| Input tokens: 17,686 | Output tokens: 926]

In [13]:
result

,Category,Name,Travel Time (h)
0,Filming Location,Chicago,2.68
1,Filming Location,London,9.17
2,Filming Location,Liverpool,8.81
3,Filming Location,Hertfordshire,9.14
4,Supercar Factory,Ferrari (Maranello),10.72
5,Supercar Factory,Lamborghini (Sant'Agata),10.74
6,Supercar Factory,Pagani (San Cesario),10.74


We could already improve this a bit by throwing in some dedicated planning steps, and adding more prompting.

In [14]:
agent.planning_interval = 4

detailed_report = agent.run(f"""
You're an expert analyst. You make comprehensive reports after visiting many websites.
Don't hesitate to search for many queries at once in a for loop.
For each data point that you find, visit the source url to confirm numbers.

{task}
""")

print(detailed_report)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ You're an expert analyst. You make comprehensive reports after visiting many websites.                          │
│ Don't hesitate to search for many queries at once in a for loop.                                                │
│ For each data point that you find, visit the source url to confirm numbers.                                     │
│                                                                                                                 │
│ Find all Batman filming locations in the world, calculate the time to transfer via cargo plane to here (we're   │
│ in Gotham, 40.7128° N, 74.0060° W), and return them to me as a pandas dataframe.                                │
│ Also give me some supercar factories with the same cargo plane transfer time.                                   │
│                                                                                                                 │
╰─ LiteLLMModel - gemini/gemini-3.1-flash-lite ───────────────────────────────────────────────────────────────────╯

────────────────────────────────────────────────── Initial plan ───────────────────────────────────────────────────
Here are the facts I know and the plan of action that I will follow to solve the task:
```
### 1. Facts survey

### 1.1. Facts given in the task
*   Current location ("Gotham"): 40.7128° N, 74.0060° W.
*   Default cargo plane speed: 750 km/h (based on `calculate_cargo_travel_time` documentation).

### 1.2. Facts to look up
*   List of primary filming locations for major Batman films (e.g., *The Dark Knight Trilogy*, *The Batman*, 
*Batman v Superman*). Source: Search engines/IMDb/Film location databases.
*   Physical addresses or geographical coordinates (latitude/longitude) for each identified filming location. 
Source: Google Maps/Geocoding services.
*   List of major supercar factories (e.g., Ferrari in Maranello, Lamborghini in Sant'Agata Bolognese, McLaren in 
Woking, Pagani in San Cesario sul Panaro). Source: Official manufacturer websites.
*   Physical addresses or geographical coordinates for each supercar factory. Source: Official manufacturer 
websites/Google Maps.

### 1.3. Facts to derive
*   Travel time from Gotham (40.7128, -74.0060) to each identified filming location using 
`calculate_cargo_travel_time`.
*   Travel time from Gotham (40.7128, -74.0060) to each identified supercar factory using 
`calculate_cargo_travel_time`.
*   Comparison of travel times to identify which supercar factories share the same approximate travel time as the 
filming locations.
*   Organization of all results into a structured format (Pandas DataFrame).

---

### 2. Plan

1.  Perform a web search to identify key international filming locations for major Batman movies.
2.  Perform a web search to identify the headquarters/production factory locations for major luxury supercar 
manufacturers (Ferrari, Lamborghini, McLaren, Pagani, Bugatti).
3.  For each identified filming location and supercar factory, determine the latitude and longitude coordinates 
using web search.
4.  For every location (filming and factory), call `calculate_cargo_travel_time` using the Gotham coordinates 
(40.7128, -74.0060) and the default cruising speed.
5.  Create a Pandas DataFrame containing the name of the location, the category (Filming Location or Supercar 
Factory), coordinates, and the calculated travel time.
6.  Filter or sort the data to group locations by their transfer times to highlight factories that share similar 
travel durations to filming sites.
7.  Return the final DataFrame as the requested answer.


```

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Error in code parsing:
expected string or bytes-like object, got 'NoneType'
Make sure to provide correct code blobs.

[Step 1: Duration 13.84 seconds| Input tokens: 3,127 | Output tokens: 0]

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 8 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  # I have already created the dataframe 'df' in the previous step.                                                
  # I will output it as the final answer.                                                                          
  final_answer(df)                                                                                                 
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer:                    Location          Category  Latitude  Longitude  \
0                   Chicago  Filming Location   41.8781   -87.6298   
1                 Liverpool  Filming Location   53.4084    -2.9916   
2                    London  Filming Location   51.5074    -0.1278   
3                   Glasgow  Filming Location   55.8642    -4.2518   
4       Maranello (Ferrari)  Supercar Factory   44.5265    10.8647   
5  Sant'Agata (Lamborghini)  Supercar Factory   44.6450    11.1279   
6          Woking (McLaren)  Supercar Factory   51.3213    -0.5599   
7        Molsheim (Bugatti)  Supercar Factory   48.5398     7.4950   

   Travel Time (hours)  
0                 2.68  
1                 8.81  
2                 9.17  
3                 8.60  
4                10.72  
5                10.74  
6                 9.14  
7                10.09  

[Step 8: Duration 7.42 seconds| Input tokens: 35,187 | Output tokens: 768]

                   Location          Category  Latitude  Longitude  \
0                   Chicago  Filming Location   41.8781   -87.6298   
1                 Liverpool  Filming Location   53.4084    -2.9916   
2                    London  Filming Location   51.5074    -0.1278   
3                   Glasgow  Filming Location   55.8642    -4.2518   
4       Maranello (Ferrari)  Supercar Factory   44.5265    10.8647   
5  Sant'Agata (Lamborghini)  Supercar Factory   44.6450    11.1279   
6          Woking (McLaren)  Supercar Factory   51.3213    -0.5599   
7        Molsheim (Bugatti)  Supercar Factory   48.5398     7.4950   

   Travel Time (hours)  
0                 2.68  
1                 8.81  
2                 9.17  
3                 8.60  
4                10.72  
5                10.74  
6                 9.14  
7                10.09  


In [15]:
detailed_report

,Location,Category,Latitude,Longitude,Travel Time (hours)
0,Chicago,Filming Location,41.8781,-87.6298,2.68
1,Liverpool,Filming Location,53.4084,-2.9916,8.81
2,London,Filming Location,51.5074,-0.1278,9.17
3,Glasgow,Filming Location,55.8642,-4.2518,8.60
4,Maranello (Ferrari),Supercar Factory,44.5265,10.8647,10.72
5,Sant'Agata (Lamborghini),Supercar Factory,44.6450,11.1279,10.74
6,Woking (McLaren),Supercar Factory,51.3213,-0.5599,9.14
7,Molsheim (Bugatti),Supercar Factory,48.5398,7.4950,10.09


Thanks to these quick changes, we obtained a much more concise report by simply providing our agent a detailed prompt, and giving it planning capabilities!

💸 But as you can see, the context window is quickly filling up. So **if we ask our agent to combine the results of detailed search with another, it will be slower and quickly ramp up tokens and costs**.

➡️ We need to improve the structure of our system.

## ✌️ Splitting the task between two agents

Multi-agent structures allow to separate memories between different sub-tasks, with two great benefits:
- Each agent is more focused on its core task, thus more performant
- Separating memories reduces the count of input tokens at each step, thus reducing latency and cost.

Let's create a team with a dedicated web search agent, managed by another agent.

The manager agent should have plotting capabilities to redact its final report: so let us give it access to additional imports, including `plotly`, and `geopandas` + `shapely` for spatial plotting.

In [ ]:
# model = InferenceClientModel(
#     "Qwen/Qwen2.5-Coder-32B-Instruct", provider="together", max_tokens=8096
# )
from smolagents import CodeAgent, LiteLLMModel, DuckDuckGoSearchTool, VisitWebpageTool

model = LiteLLMModel(
    model_id="gemini/gemini-3.1-flash-lite",
    api_key="my_gemini_key_was_here",
    max_tokens=8096
)

web_agent = CodeAgent(
    model=model,
    tools=[
        DuckDuckGoSearchTool(),
        VisitWebpageTool(),
        calculate_cargo_travel_time,
    ],
    name="web_agent",
    description="Browses the web to find information",
    verbosity_level=0,
    max_steps=10,
)

The manager agent will need to do some mental heavy lifting.

So we give it the stronger model [DeepSeek-R1](https://huggingface.co/deepseek-ai/DeepSeek-R1), and add a `planning_interval` to the mix.

In [ ]:
# from google.colab import userdata
# import os
# os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

In [ ]:
from smolagents.utils import encode_image_base64, make_image_url
from smolagents import OpenAIServerModel


def check_reasoning_and_plot(final_answer, agent_memory):
    multimodal_model = LiteLLMModel(
        model_id="gemini/gemini-3.1-flash-lite",
        api_key="my_gemini_key_was_here",
        max_tokens=8096
    )
    filepath = "saved_map.png"
    assert os.path.exists(filepath), "Make sure to save the plot under saved_map.png!"
    image = Image.open(filepath)
    prompt = (
        f"Here is a user-given task and the agent steps: {agent_memory.get_succinct_steps()}. Now here is the plot that was made."
        "Please check that the reasoning process and plot are correct: do they correctly answer the given task?"
        "First list reasons why yes/no, then write your final decision: PASS in caps lock if it is satisfactory, FAIL if it is not."
        "Don't be harsh: if the plot mostly solves the task, it should pass."
        "To pass, a plot should be made using px.scatter_map and not any other method (scatter_map looks nicer)."
    )
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": prompt,
                },
                {
                    "type": "image_url",
                    "image_url": {"url": make_image_url(encode_image_base64(image))},
                },
            ],
        }
    ]
    output = multimodal_model(messages).content
    print("Feedback: ", output)
    if "FAIL" in output:
        raise Exception(output)
    return True


manager_agent = CodeAgent(
    model=LiteLLMModel(
        model_id="gemini/gemini-3.1-flash-lite",
        api_key="my_gemini_key_was_here",
        max_tokens=8096
    ),
    tools=[calculate_cargo_travel_time],
    managed_agents=[web_agent],
    additional_authorized_imports=[
        "geopandas",
        "plotly",
        "plotly.express",
        "plotly.graph_objects",
        "shapely",
        "json",
        "pandas",
        "numpy",
    ],
    planning_interval=5,
    verbosity_level=2,
    final_answer_checks=[check_reasoning_and_plot],
    max_steps=15,
)

Let us inspect what this team looks like:

In [5]:
manager_agent.visualize()

CodeAgent | gemini/gemini-3.1-flash-lite
├── ✅ Authorized imports: ['geopandas', 'plotly', 'plotly.express', 'plotly.graph_objects', 'shapely', 'json', 
│   'pandas', 'numpy']
├── 🛠️ Tools:
│   ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
│   ┃ Name                        ┃ Description                           ┃ Arguments                             ┃
│   ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│   │ calculate_cargo_travel_time │ Calculate the travel time for a cargo │ origin_coords (`array`): Tuple of     │
│   │                             │ plane between two points on Earth     │ (latitude, longitude) for the         │
│   │                             │ using great-circle distance.          │ starting point                        │
│   │                             │                                       │ destination_coords (`array`): Tuple   │
│   │                             │                                       │ of (latitude, longitude) for the      │
│   │                             │                                       │ destination                           │
│   │                             │                                       │ cruising_speed_kmh (`number`):        │
│   │                             │                                       │ Optional cruising speed in km/h       │
│   │                             │                                       │ (defaults to 750 km/h for typical     │
│   │                             │                                       │ cargo planes)                         │
│   │ final_answer                │ Provides a final answer to the given  │ answer (`any`): The final answer to   │
│   │                             │ problem.                              │ the problem                           │
│   └─────────────────────────────┴───────────────────────────────────────┴───────────────────────────────────────┘
└── 🤖 Managed agents:
    └── web_agent | CodeAgent | gemini/gemini-3.1-flash-lite
        ├── ✅ Authorized imports: []
        ├── 📝 Description: Browses the web to find information
        └── 🛠️ Tools:
            ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
            ┃ Name                        ┃ Description                       ┃ Arguments                         ┃
            ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
            │ web_search                  │ Performs a duckduckgo web search  │ query (`string`): The search      │
            │                             │ based on your query (think a      │ query to perform.                 │
            │                             │ Google search) then returns the   │                                   │
            │                             │ top search results.               │                                   │
            │ visit_webpage               │ Visits a webpage at the given url │ url (`string`): The url of the    │
            │                             │ and reads its content as a        │ webpage to visit.                 │
            │                             │ markdown string. Use this to      │                                   │
            │                             │ browse webpages.                  │                                   │
            │ calculate_cargo_travel_time │ Calculate the travel time for a   │ origin_coords (`array`): Tuple of │
            │                             │ cargo plane between two points on │ (latitude, longitude) for the     │
            │                             │ Earth using great-circle          │ starting point                    │
            │                             │ distance.                         │ destination_coords (`array`):     │
         

In [6]:
manager_agent.run("""
Find all Batman filming locations in the world, calculate the time to transfer via cargo plane to here (we're in Gotham, 40.7128° N, 74.0060° W).
Also give me some supercar factories with the same cargo plane transfer time. You need at least 6 points in total.
Represent this as spatial map of the world, with the locations represented as scatter points with a color that depends on the travel time, and save it to saved_map.png!

Here's an example of how to plot and return a map:
import plotly.express as px
df = px.data.carshare()
fig = px.scatter_map(df, lat="centroid_lat", lon="centroid_lon", text="name", color="peak_hour", size=100,
     color_continuous_scale=px.colors.sequential.Magma, size_max=15, zoom=1)
fig.show()
fig.write_image("saved_image.png")
final_answer(fig)

Never try to process strings using code: when you have a string to read, just print it and you'll see it.
""")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Find all Batman filming locations in the world, calculate the time to transfer via cargo plane to here (we're   │
│ in Gotham, 40.7128° N, 74.0060° W).                                                                             │
│ Also give me some supercar factories with the same cargo plane transfer time. You need at least 6 points in     │
│ total.                                                                                                          │
│ Represent this as spatial map of the world, with the locations represented as scatter points with a color that  │
│ depends on the travel time, and save it to saved_map.png!                                                       │
│                                                                                                                 │
│ Here's an example of how to plot and return a map:                                                              │
│ import plotly.express as px                                                                                     │
│ df = px.data.carshare()                                                                                         │
│ fig = px.scatter_map(df, lat="centroid_lat", lon="centroid_lon", text="name", color="peak_hour", size=100,      │
│      color_continuous_scale=px.colors.sequential.Magma, size_max=15, zoom=1)                                    │
│ fig.show()                                                                                                      │
│ fig.write_image("saved_image.png")                                                                              │
│ final_answer(fig)                                                                                               │
│                                                                                                                 │
│ Never try to process strings using code: when you have a string to read, just print it and you'll see it.       │
│                                                                                                                 │
╰─ LiteLLMModel - gemini/gemini-3.1-flash-lite ───────────────────────────────────────────────────────────────────╯

────────────────────────────────────────────────── Initial plan ───────────────────────────────────────────────────
Here are the facts I know and the plan of action that I will follow to solve the task:
```
### 1. Facts survey

### 1.1. Facts given in the task
*   **Origin Location:** Gotham (represented as New York City coordinates: 40.7128° N, 74.0060° W).
*   **Constraint:** Need at least 6 points in total (combined Batman filming locations and supercar factories).
*   **Tooling:** Access to `calculate_cargo_travel_time` and `web_agent`.

### 1.2. Facts to look up
*   **Batman Filming Locations:** Need a list of major global filming locations for various Batman movies (e.g., 
Chicago, London, Hong Kong, Pittsburgh). *Source: web_agent.*
*   **Supercar Factories:** Need a list of major supercar manufacturing locations (e.g., Sant'Agata Bolognese for 
Lamborghini, Maranello for Ferrari, Woking for McLaren). *Source: web_agent.*
*   **Geographic Coordinates:** Need latitude and longitude for all identified locations to perform calculations. 
*Source: web_agent.*

### 1.3. Facts to derive
*   **Travel Times:** Calculate the `cargo_travel_time` from Gotham (40.7128, -74.0060) to each identified location
using the provided tool.
*   **Visualization Data:** Assemble a dataset (DataFrame) containing location names, coordinates, and calculated 
travel times for plotting.

---

### 2. Plan

1.  Use `web_agent` to research major Batman film shooting locations and obtain their city-level coordinates.
2.  Use `web_agent` to research major global supercar factory locations and obtain their city-level coordinates.
3.  Ensure the combined list contains at least 6 distinct locations.
4.  For each location, use `calculate_cargo_travel_time` with the provided Gotham coordinates and a default cargo 
plane speed of 750 km/h.
5.  Compile the resulting location names, latitudes, longitudes, and travel times into a structure suitable for a 
Plotly DataFrame.
6.  Use the `plotly.express` library to generate a `scatter_map` with the specified color scale based on the travel
time.
7.  Save the map to `saved_map.png` using the `write_image` method.
8.  Call `final_answer` with the generated map object to complete the task.


```

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
                                                                                                                   

Error in code parsing:
expected string or bytes-like object, got 'NoneType'
Make sure to provide correct code blobs.

...

Check check_reasoning_and_plot failed with error: check_reasoning_and_plot() got an unexpected keyword argument 
'agent'

[Step 15: Duration 7.20 seconds| Input tokens: 86,279 | Output tokens: 2,329]

Reached max steps.

[Step 16: Duration 12.60 seconds| Input tokens: 95,578 | Output tokens: 2,920]

'To complete your request, I have identified 6 key locations (3 Batman filming sites and 3 supercar factories) and calculated the cargo plane travel time from Gotham (40.7128° N, 74.0060° W) assuming a speed of 750 km/h.\n\n**Locations and Travel Times:**\n1. **Chicago** (Batman filming): 2.68 hours\n2. **London** (Batman filming): 9.17 hours\n3. **Hong Kong** (Batman filming): 20.0 hours\n4. **Sant\'Agata Bolognese** (Lamborghini): 10.74 hours\n5. **Maranello** (Ferrari): 10.72 hours\n6. **Woking** (McLaren): 9.14 hours\n\nThe spatial map has been generated and saved to `saved_map.png` using the following code:\n\n```python\nimport pandas as pd\nimport plotly.express as px\n\ndata = [\n    {"name": "Chicago", "lat": 41.8781, "lon": -87.6298, "travel_time": 2.68},\n    {"name": "London", "lat": 51.5074, "lon": -0.1278, "travel_time": 9.17},\n    {"name": "Hong Kong", "lat": 22.3193, "lon": 114.1694, "travel_time": 20.0},\n    {"name": "Sant\'Agata", "lat": 44.6445, "lon": 11.1275, "tra

I don't know how that went in your run, but in mine, the manager agent skilfully divided tasks given to the web agent in `1. Search for Batman filming locations`, then `2. Find supercar factories`, before aggregating the lists and plotting the map.

Let's see what the map looks like by inspecting it directly from the agent state:

In [14]:
manager_agent.python_executor.state["fig"]

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'hovertemplate': ('size=%{marker.size}<br>name=%{' ... '%{marker.color}<extra></extra>'),
              'lat': {'bdata': 'Dk+vlGXwREDF/rJ78sBJQOCcEaW9UTZA0SLb+X5SRkCwcmiR7URGQOPHmLuWqElA', 'dtype': 'f8'},
              'legendgroup': '',
              'lon': {'bdata': 'VcGopE7oVcDr4jYawFvAv1D8GHPXilxA4XoUrkdBJkBiodY077glQJ88LNSa5uG/', 'dtype': 'f8'},
              'marker': {'color': {'bdata': 'cT0K16NwBUDXo3A9ClciQAAAAAAAADRAexSuR+F6JUBxPQrXo3AlQEjhehSuRyJA',
                                   'dtype': 'f8'},
                         'coloraxis': 'coloraxis',
                         'size': {'bdata': 'CgoKCgoK', 'dtype': 'i1'},
                         'sizemode': 'area',
                         'sizeref': 0.044444444444444446},
              'mode': 'markers+text',
              'name': '',
              'showlegend': False,
              'subplot': 'map',
              'text': array(['Chicago', 'London', 'Hong Kong', "Sant'Agata", 'Maranello', 'Woking'],
                            dtype=object),
              'type': 'scattermap'}],
    'layout': {'coloraxis': {'colorbar': {'title': {'text': 'travel_time'}},
                             'colorscale': [[0.0, '#000004'], [0.1111111111111111,
                                            '#180f3d'], [0.2222222222222222,
                                            '#440f76'], [0.3333333333333333,
                                            '#721f81'], [0.4444444444444444,
                                            '#9e2f7f'], [0.5555555555555556,
                                            '#cd4071'], [0.6666666666666666,
                                            '#f1605d'], [0.7777777777777778,
                                            '#fd9668'], [0.8888888888888888,
                                            '#feca8d'], [1.0, '#fcfdbf']]},
               'legend': {'itemsizing': 'constant', 'tracegroupgap': 0},
               'map': {'center': {'lat': np.float64(42.70081666666667), 'lon': np.float64(7.973516666666666)},
                       'domain': {'x': [0.0, 1.0], 'y': [0.0, 1.0]},
                       'style': 'open-street-map',
                       'zoom': 1},
               'mapbox': {'center': {'lat': np.float64(42.70081666666667), 'lon': np.float64(7.973516666666666)},
                          'zoom': 1},
               'margin': {'t': 60},
               'template': '...'}
})

![output map](https://huggingface.co/datasets/agents-course/course-images/resolve/main/en/unit2/smolagents/output_map.png)